# Normalización completa del dataset de Política

## 1. Importar librerías necesarias

In [1]:
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path
from difflib import get_close_matches

## 2. Cargar el dataset de política a normalizar

In [2]:
# Definir la ruta del dataset de Política normalizado (generado por el notebook anterior)
dataset_path = r'C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\08 - Politica\01 - politica municipios normalizados.csv'

# Cargar el archivo CSV
df = pd.read_csv(dataset_path)

print(f'Filas cargadas: {len(df)}')
print('Columnas disponibles:', list(df.columns))
print(f'Tamaño del dataset: {df.shape[0]} filas x {df.shape[1]} columnas')

# Mostrar primeras filas
print(f"\nPRIMERAS FILAS DEL DATASET:")
df.head()

Filas cargadas: 2101
Columnas disponibles: ['comunidad_autonoma', 'provincia', 'municipio', 'partido_politico', 'fecha_posesion', 'fecha_baja']
Tamaño del dataset: 2101 filas x 6 columnas

PRIMERAS FILAS DEL DATASET:


,comunidad_autonoma,provincia,municipio,partido_politico,fecha_posesion,fecha_baja
0,Galicia,Lugo ...,abadín,PP,03/07/1999,NaN
1,Galicia,Lugo ...,abadín,PP,14/06/2003,NaN
2,Galicia,"Coruña, A ...",abegondo,PP,03/07/1999,14/06/2003
3,Galicia,"Coruña, A ...",abegondo,PP,14/06/2003,27/09/2004
4,Galicia,"Coruña, A ...",abegondo,PP,14/06/2003,27/09/2004


In [3]:
# Verificar la estructura del dataset de Política normalizado
print("=" * 60)
print("VERIFICACIÓN DE ESTRUCTURA DEL DATASET POLÍTICA")
print("=" * 60)

# El dataset viene del notebook anterior, verificar columnas actuales
columnas_presentes = df.columns.tolist()

print(f"Columnas encontradas: {len(columnas_presentes)}")
for i, col in enumerate(columnas_presentes, 1):
    print(f"   {i}. {col}")

# Verificar tipos de datos
print(f"\nTIPOS DE DATOS:")
print("-" * 30)
for col in columnas_presentes:
    tipo = df[col].dtype
    nulos = df[col].isnull().sum()
    unicos = df[col].nunique() if df[col].dtype != 'object' or len(df) < 1000 else 'muchos'
    print(f"   • {col}: {tipo} ({nulos:,} nulos, {unicos} únicos)")

# Verificar información temporal si existe
if 'fecha posesión' in df.columns:
    print(f"\nINFORMACIÓN TEMPORAL (fecha posesión):")
    print(f"   • Tipo actual: {df['fecha posesión'].dtype}")
    print(f"   • Valores únicos: {df['fecha posesión'].nunique():,}")
    print(f"   • Ejemplos: {df['fecha posesión'].head(3).tolist()}")
elif 'fecha_posesion' in df.columns:
    print(f"\nINFORMACIÓN TEMPORAL (fecha_posesion):")
    print(f"   • Tipo actual: {df['fecha_posesion'].dtype}")
    print(f"   • Valores únicos: {df['fecha_posesion'].nunique():,}")
    print(f"   • Ejemplos: {df['fecha_posesion'].head(3).tolist()}")

if 'fecha baja' in df.columns:
    print(f"\nINFORMACIÓN TEMPORAL (fecha baja):")
    print(f"   • Tipo actual: {df['fecha baja'].dtype}")
    print(f"   • Valores únicos: {df['fecha baja'].nunique():,}")
    print(f"   • Nulos: {df['fecha baja'].isnull().sum():,}")
elif 'fecha_baja' in df.columns:
    print(f"\nINFORMACIÓN TEMPORAL (fecha_baja):")
    print(f"   • Tipo actual: {df['fecha_baja'].dtype}")
    print(f"   • Valores únicos: {df['fecha_baja'].nunique():,}")
    print(f"   • Nulos: {df['fecha_baja'].isnull().sum():,}")

print("=" * 60)

VERIFICACIÓN DE ESTRUCTURA DEL DATASET POLÍTICA
Columnas encontradas: 6
   1. comunidad_autonoma
   2. provincia
   3. municipio
   4. partido_politico
   5. fecha_posesion
   6. fecha_baja

TIPOS DE DATOS:
------------------------------
   • comunidad_autonoma: object (0 nulos, muchos únicos)
   • provincia: object (0 nulos, muchos únicos)
   • municipio: object (0 nulos, muchos únicos)
   • partido_politico: object (0 nulos, muchos únicos)
   • fecha_posesion: object (0 nulos, muchos únicos)
   • fecha_baja: object (685 nulos, muchos únicos)

INFORMACIÓN TEMPORAL (fecha_posesion):
   • Tipo actual: object
   • Valores únicos: 229
   • Ejemplos: ['03/07/1999', '14/06/2003', '03/07/1999']

INFORMACIÓN TEMPORAL (fecha_baja):
   • Tipo actual: object
   • Valores únicos: 214
   • Nulos: 685


In [4]:
# Normalizar nombres de columnas del dataset político
print("NORMALIZACIÓN DE NOMBRES DE COLUMNAS")
print("-" * 45)

# Mapeo de columnas del dataset político real (ya normalizado en el paso anterior)
mapeo_columnas_politica = {
    'comunidad_autonoma': 'comunidad_autonoma',
    'provincia': 'provincia',
    'municipio': 'municipio',
    'partido_politico': 'partido',
    'fecha_posesion': 'fecha_posesion',
    'fecha_baja': 'fecha_baja'
}

print(f"Columnas originales encontradas: {len(df.columns)}")
print(f"Aplicando mapeo de columnas:")

# Mostrar mapeo y aplicar renombrado
columnas_renombradas = 0
for original, nuevo in mapeo_columnas_politica.items():
    if original in df.columns:
        print(f"   '{original}' → '{nuevo}'")
        columnas_renombradas += 1
    else:
        print(f"   '{original}' → '{nuevo}' (columna no encontrada)")

# Aplicar el mapeo
df.rename(columns=mapeo_columnas_politica, inplace=True)

# Seleccionar solo las columnas importantes para el análisis
columnas_mantener = [
    'municipio', 'partido', 'fecha_posesion', 
    'fecha_baja', 'provincia', 'comunidad_autonoma'
]

# Filtrar solo las columnas que realmente existen
columnas_disponibles = [col for col in columnas_mantener if col in df.columns]
columnas_faltantes = [col for col in columnas_mantener if col not in df.columns]

print(f"\nRESULTADO DEL MAPEO:")
print(f"   • Columnas renombradas: {columnas_renombradas}")
print(f"   • Columnas disponibles: {len(columnas_disponibles)}")
print(f"   • Columnas finales: {columnas_disponibles}")

if columnas_faltantes:
    print(f"   Columnas faltantes: {columnas_faltantes}")

# Mantener solo las columnas importantes
df = df[columnas_disponibles].copy()

print(f"\nDATASET FINAL:")
print(f"   • Registros: {len(df):,}")
print(f"   • Columnas: {len(df.columns)}")
for i, col in enumerate(df.columns, 1):
    print(f"   {i}. {col} ({df[col].dtype})")

NORMALIZACIÓN DE NOMBRES DE COLUMNAS
---------------------------------------------
Columnas originales encontradas: 6
Aplicando mapeo de columnas:
   'comunidad_autonoma' → 'comunidad_autonoma'
   'provincia' → 'provincia'
   'municipio' → 'municipio'
   'partido_politico' → 'partido'
   'fecha_posesion' → 'fecha_posesion'
   'fecha_baja' → 'fecha_baja'

RESULTADO DEL MAPEO:
   • Columnas renombradas: 6
   • Columnas disponibles: 6
   • Columnas finales: ['municipio', 'partido', 'fecha_posesion', 'fecha_baja', 'provincia', 'comunidad_autonoma']

DATASET FINAL:
   • Registros: 2,101
   • Columnas: 6
   1. municipio (object)
   2. partido (object)
   3. fecha_posesion (object)
   4. fecha_baja (object)
   5. provincia (object)
   6. comunidad_autonoma (object)


In [5]:
# Convertir fechas de posesión y baja a tipo datetime estandarizado
print("NORMALIZANDO COLUMNAS DE FECHA")
print("-" * 40)

# Procesar fecha de posesión
if 'fecha_posesion' in df.columns:
    print(f"Procesando 'fecha_posesion'")
    print(f"Tipo actual: {df['fecha_posesion'].dtype}")
    print(f"Ejemplos antes: {df['fecha_posesion'].head(3).tolist()}")
    
    # Convertir a datetime (maneja múltiples formatos)
    df['fecha_posesion'] = pd.to_datetime(df['fecha_posesion'], errors='coerce', dayfirst=True)
    
    # Verificar valores nulos después de conversión
    nulos_posesion = df['fecha_posesion'].isnull().sum()
    if nulos_posesion > 0:
        print(f"{nulos_posesion:,} fechas de posesión no pudieron convertirse")
    
    print(f"fecha_posesion convertida a datetime")
    
    # Mostrar rango solo si hay fechas válidas
    fechas_validas = df['fecha_posesion'].dropna()
    if len(fechas_validas) > 0:
        print(f"Rango completo: {fechas_validas.min()} a {fechas_validas.max()}")
        print(f"Ejemplos después: {df['fecha_posesion'].head(3).tolist()}")
    else:
        print("No hay fechas válidas")
else:
    print("Columna 'fecha_posesion' no encontrada")

# Procesar fecha de baja
if 'fecha_baja' in df.columns:
    print(f"\nProcesando 'fecha_baja'")
    print(f"Tipo actual: {df['fecha_baja'].dtype}")
    
    # Verificar valores nulos antes
    nulos_antes = df['fecha_baja'].isnull().sum()
    print(f"Valores nulos antes: {nulos_antes:,}")
    
    # Convertir a datetime (maneja múltiples formatos)
    df['fecha_baja'] = pd.to_datetime(df['fecha_baja'], errors='coerce', dayfirst=True)
    
    # Verificar valores nulos después de conversión
    nulos_despues = df['fecha_baja'].isnull().sum()
    if nulos_despues > nulos_antes:
        print(f"{nulos_despues - nulos_antes:,} fechas de baja adicionales no pudieron convertirse")
    
    print(f"fecha_baja convertida a datetime")
    print(f"Valores nulos después: {nulos_despues:,}")
    
    # Mostrar rango solo si hay fechas válidas
    fechas_baja_validas = df['fecha_baja'].dropna()
    if len(fechas_baja_validas) > 0:
        print(f"Rango (sin nulos): {fechas_baja_validas.min()} a {fechas_baja_validas.max()}")
    else:
        print("No hay fechas de baja válidas")
else:
    print("Columna 'fecha_baja' no encontrada")

# Resumen de fechas
print(f"\nRESUMEN DE FECHAS:")
if 'fecha_posesion' in df.columns:
    print(f"   • Fechas de posesión únicas: {df['fecha_posesion'].nunique():,}")
    print(f"   • Fechas de posesión nulas: {df['fecha_posesion'].isnull().sum():,}")
if 'fecha_baja' in df.columns:
    print(f"   • Fechas de baja únicas: {df['fecha_baja'].nunique():,}")
    print(f"   • Fechas de baja nulas: {df['fecha_baja'].isnull().sum():,}")

NORMALIZANDO COLUMNAS DE FECHA
----------------------------------------
Procesando 'fecha_posesion'
Tipo actual: object
Ejemplos antes: ['03/07/1999', '14/06/2003', '03/07/1999']
fecha_posesion convertida a datetime
Rango completo: 1999-07-03 00:00:00 a 2023-01-23 00:00:00
Ejemplos después: [Timestamp('1999-07-03 00:00:00'), Timestamp('2003-06-14 00:00:00'), Timestamp('1999-07-03 00:00:00')]

Procesando 'fecha_baja'
Tipo actual: object
Valores nulos antes: 685
fecha_baja convertida a datetime
Valores nulos después: 685
Rango (sin nulos): 1999-11-12 00:00:00 a 2023-01-23 00:00:00

RESUMEN DE FECHAS:
   • Fechas de posesión únicas: 229
   • Fechas de posesión nulas: 0
   • Fechas de baja únicas: 214
   • Fechas de baja nulas: 685


In [6]:
# Filtrar registros solo entre el 1 de enero de 2000 y el 31 de diciembre de 2022
print("FILTRANDO PERÍODO TEMPORAL (2000-2022)")
print("-" * 45)

fecha_inicio = pd.to_datetime('2000-01-01')
fecha_fin = pd.to_datetime('2022-12-31')

print(f"Registros antes del filtro: {len(df):,}")
print(f"Período a mantener: {fecha_inicio.date()} a {fecha_fin.date()}")

# Aplicar filtro basado en fecha de posesión
if 'fecha_posesion' in df.columns:
    # Crear máscara de filtrado
    mask_fecha = (df['fecha_posesion'] >= fecha_inicio) & (df['fecha_posesion'] <= fecha_fin)
    df_filtrado = df[mask_fecha].copy()
    
    # Mostrar estadísticas del filtrado
    registros_eliminados = len(df) - len(df_filtrado)
    print(f"Registros después del filtro: {len(df_filtrado):,}")
    print(f"Registros eliminados: {registros_eliminados:,}")
    
    if registros_eliminados > 0:
        porcentaje_conservado = (len(df_filtrado)/len(df)*100)
        print(f"Porcentaje conservado: {porcentaje_conservado:.2f}%")
        
        # Mostrar qué fechas se eliminaron
        fechas_eliminadas = df[~mask_fecha]['fecha_posesion'].dropna()
        if len(fechas_eliminadas) > 0:
            fecha_min_eliminada = fechas_eliminadas.min()
            fecha_max_eliminada = fechas_eliminadas.max()
            print(f"Fechas eliminadas van de: {fecha_min_eliminada.date()} a {fecha_max_eliminada.date()}")
    
    # Actualizar el dataframe
    df = df_filtrado
    
    print(f"\nFiltrado completado")
    if len(df) > 0:
        fecha_min_final = df['fecha_posesion'].min()
        fecha_max_final = df['fecha_posesion'].max()
        print(f"Rango final de posesiones: {fecha_min_final.date()} a {fecha_max_final.date()}")
        print(f"Registros finales: {len(df):,}")
        print(f"Municipios únicos: {df['municipio'].nunique()}")
    else:
        print("No quedan registros después del filtrado")
else:
    print("No se encontró columna 'fecha_posesion' para filtrar")

FILTRANDO PERÍODO TEMPORAL (2000-2022)
---------------------------------------------
Registros antes del filtro: 2,101
Período a mantener: 2000-01-01 a 2022-12-31
Registros después del filtro: 1,784
Registros eliminados: 317
Porcentaje conservado: 84.91%
Fechas eliminadas van de: 1999-07-03 a 2023-01-23

Filtrado completado
Rango final de posesiones: 2000-02-05 a 2022-11-18
Registros finales: 1,784
Municipios únicos: 315


In [7]:
# Cargar información de Presidentes de la Xunta de Galicia y ajustar períodos
print("CARGANDO INFORMACIÓN DE PRESIDENTES AUTONÓMICOS")
print("-" * 55)

# Cargar el archivo Excel con los presidentes autonómicos
presidentes_file = r'C:\00 - Proyecto Incendios Galicia - END\data\01 - raw\08 - politica\07 - Presidentes autonomicos.xlsx'
df_presidentes = pd.read_excel(presidentes_file)

print(f"Archivo cargado: {presidentes_file}")
print(f"Columnas encontradas: {list(df_presidentes.columns)}")
print(f"Filas: {len(df_presidentes)}")

# Normalizar nombres de columnas
df_presidentes.rename(columns={
    'Fecha inicio mandato': 'fecha_inicio',
    'fecha fin mandato': 'fecha_fin',
    'partido politico autonomico': 'partido_autonomico'
}, inplace=True)

# Convertir fechas a datetime
df_presidentes['fecha_inicio'] = pd.to_datetime(df_presidentes['fecha_inicio'])
df_presidentes['fecha_fin'] = pd.to_datetime(df_presidentes['fecha_fin'])

print(f"\nPERÍODOS ORIGINALES DE PRESIDENTES AUTONÓMICOS:")
print("-" * 55)
for i, row in df_presidentes.iterrows():
    fecha_inicio = row['fecha_inicio'].date()
    fecha_fin = row['fecha_fin'].date()
    partido = row['partido_autonomico']
    print(f"   {i+1}. {partido}: {fecha_inicio} a {fecha_fin}")

# CORRECCIÓN IMPORTANTE: Ajustar períodos para evitar solapamientos
# El problema es que las fechas se solapan exactamente en las transiciones
print(f"\nAJUSTANDO PERÍODOS PARA EVITAR SOLAPAMIENTOS:")
print("-" * 50)

# Ajustar las fechas fin para que no se solapen con las fechas inicio del siguiente
df_presidentes_ajustado = df_presidentes.copy()

# Para cada fila excepto la última, ajustar fecha_fin un día antes del siguiente inicio
for i in range(len(df_presidentes_ajustado) - 1):
    fecha_siguiente_inicio = df_presidentes_ajustado.iloc[i + 1]['fecha_inicio']
    fecha_fin_ajustada = fecha_siguiente_inicio - pd.Timedelta(days=1)
    df_presidentes_ajustado.iloc[i, df_presidentes_ajustado.columns.get_loc('fecha_fin')] = fecha_fin_ajustada

print(f"PERÍODOS AJUSTADOS (sin solapamientos):")
print("-" * 45)
for i, row in df_presidentes_ajustado.iterrows():
    fecha_inicio = row['fecha_inicio'].date()
    fecha_fin = row['fecha_fin'].date()
    partido = row['partido_autonomico']
    print(f"   {i+1}. {partido}: {fecha_inicio} a {fecha_fin}")

# Verificar rango de fechas de alcaldes vs presidentes
if len(df) > 0 and 'fecha_posesion' in df.columns:
    fechas_alcaldes = df['fecha_posesion'].dropna()
    if len(fechas_alcaldes) > 0:
        min_alcalde = fechas_alcaldes.min()
        max_alcalde = fechas_alcaldes.max()
        min_presidente = df_presidentes_ajustado['fecha_inicio'].min()
        max_presidente = df_presidentes_ajustado['fecha_fin'].max()
        
        print(f"\nCOMPARACIÓN DE RANGOS:")
        print(f"   • Alcaldes: {min_alcalde.date()} a {max_alcalde.date()}")
        print(f"   • Presidentes: {min_presidente.date()} a {max_presidente.date()}")
        
        # Verificar cobertura
        if min_alcalde < min_presidente:
            print(f"   Alcaldes anteriores a primer presidente: {min_alcalde.date()}")
        if max_alcalde > max_presidente:
            print(f"   Alcaldes posteriores a último presidente: {max_alcalde.date()}")

# Función para asignar partido autonómico según la fecha (usando períodos ajustados)
def asignar_presidente_autonomico(fecha_posesion):
    """Asigna el partido autonómico según la fecha de posesión del alcalde"""
    if pd.isna(fecha_posesion):
        return None
    
    # Buscar en qué período cae la fecha (usando períodos ajustados)
    for _, periodo in df_presidentes_ajustado.iterrows():
        if periodo['fecha_inicio'] <= fecha_posesion <= periodo['fecha_fin']:
            return periodo['partido_autonomico']
    
    return None  # Si no se encuentra en ningún período

# Aplicar la función a cada registro
print(f"\nAsignando partidos autonómicos a {len(df):,} registros...")
df['partido_autonomico'] = df['fecha_posesion'].apply(asignar_presidente_autonomico)

# Verificar resultados
partidos_asignados = df['partido_autonomico'].notna().sum()
partidos_nulos = df['partido_autonomico'].isnull().sum()

print(f"\nRESULTADOS DE ASIGNACIÓN:")
print(f"   • Partidos autonómicos asignados: {partidos_asignados:,}")
print(f"   • Registros sin asignar: {partidos_nulos:,}")
print(f"   • Cobertura: {(partidos_asignados/len(df)*100):.1f}%")

# Mostrar distribución de partidos autonómicos
if partidos_asignados > 0:
    print(f"\nDISTRIBUCIÓN DE PARTIDOS AUTONÓMICOS:")
    dist_partidos_aut = df['partido_autonomico'].value_counts()
    for partido, count in dist_partidos_aut.items():
        porcentaje = (count / partidos_asignados) * 100
        print(f"   • {partido}: {count:,} registros ({porcentaje:.1f}%)")

# Si hay registros sin asignar, mostrar sus fechas para diagnóstico
if partidos_nulos > 0:
    print(f"\nFECHAS SIN ASIGNAR:")
    fechas_sin_asignar = df[df['partido_autonomico'].isnull()]['fecha_posesion'].dropna()
    if len(fechas_sin_asignar) > 0:
        fecha_min_sin = fechas_sin_asignar.min()
        fecha_max_sin = fechas_sin_asignar.max()
        print(f"   • Rango: {fecha_min_sin.date()} a {fecha_max_sin.date()}")
        print(f"   • Ejemplos: {[f.date() for f in fechas_sin_asignar.head(5)]}")

print(f"\nInformación de presidentes autonómicos añadida exitosamente")

CARGANDO INFORMACIÓN DE PRESIDENTES AUTONÓMICOS
-------------------------------------------------------
Archivo cargado: C:\00 - Proyecto Incendios Galicia - END\data\01 - raw\08 - politica\07 - Presidentes autonomicos.xlsx
Columnas encontradas: ['Fecha inicio mandato', 'fecha fin mandato', 'partido politico autonomico']
Filas: 4

PERÍODOS ORIGINALES DE PRESIDENTES AUTONÓMICOS:
-------------------------------------------------------
   1. PP: 1990-02-01 a 2005-08-02
   2. PSOE: 2005-08-02 a 2009-04-16
   3. PP: 2009-04-16 a 2022-05-14
   4. PP: 2022-05-14 a 2025-08-06

AJUSTANDO PERÍODOS PARA EVITAR SOLAPAMIENTOS:
--------------------------------------------------
PERÍODOS AJUSTADOS (sin solapamientos):
---------------------------------------------
   1. PP: 1990-02-01 a 2005-08-01
   2. PSOE: 2005-08-02 a 2009-04-15
   3. PP: 2009-04-16 a 2022-05-13
   4. PP: 2022-05-14 a 2025-08-06

COMPARACIÓN DE RANGOS:
   • Alcaldes: 2000-02-05 a 2022-11-18
   • Presidentes: 1990-02-01 a 2025-08-0

In [8]:
# Preparar y guardar el dataset final
print("PREPARANDO DATASET FINAL PARA EXPORTACIÓN")
print("-" * 50)

# Crear copia para exportación
df_export = df.copy()

# Convertir fechas datetime a formato string YYYY-MM-DD para compatibilidad
if 'fecha_posesion' in df_export.columns:
    df_export['fecha_posesion'] = df_export['fecha_posesion'].dt.strftime('%Y-%m-%d')
    print("fecha_posesion convertida a formato string")

if 'fecha_baja' in df_export.columns:
    # Manejar valores nulos en fecha_baja
    df_export['fecha_baja'] = df_export['fecha_baja'].dt.strftime('%Y-%m-%d').fillna('')
    print("fecha_baja convertida a formato string (nulos = '')")

# Crear directorio de destino
import os
ruta_export = r'C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\08 - Politica'
os.makedirs(ruta_export, exist_ok=True)

# Definir archivo de salida
archivo_export = os.path.join(ruta_export, '01 - politica normalizado completo.csv')

# Guardar dataset
df_export.to_csv(archivo_export, index=False, encoding='utf-8')

print(f"\nARCHIVO EXPORTADO:")
print(f"   • Ubicación: {archivo_export}")
print(f"   • Formato: CSV UTF-8")
print(f"   • Registros: {len(df_export):,}")
print(f"   • Columnas: {len(df_export.columns)}")

# Verificar fechas en el dataset final
if len(df_export) > 0:
    fecha_min = df_export['fecha_posesion'].min()
    fecha_max = df_export['fecha_posesion'].max()
    print(f"   • Período: {fecha_min} a {fecha_max}")

print(f"   • Columnas finales: {list(df_export.columns)}")

# Verificar que el archivo se guardó correctamente
if os.path.exists(archivo_export):
    tamaño_archivo = os.path.getsize(archivo_export)
    print(f"   • Tamaño del archivo: {tamaño_archivo:,} bytes")
    print("Archivo guardado exitosamente")
else:
    print("Error al guardar el archivo")

PREPARANDO DATASET FINAL PARA EXPORTACIÓN
--------------------------------------------------
fecha_posesion convertida a formato string
fecha_baja convertida a formato string (nulos = '')

ARCHIVO EXPORTADO:
   • Ubicación: C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\08 - Politica\01 - politica normalizado completo.csv
   • Formato: CSV UTF-8
   • Registros: 1,784
   • Columnas: 7
   • Período: 2000-02-05 a 2022-11-18
   • Columnas finales: ['municipio', 'partido', 'fecha_posesion', 'fecha_baja', 'provincia', 'comunidad_autonoma', 'partido_autonomico']
   • Tamaño del archivo: 112,533 bytes
Archivo guardado exitosamente


In [9]:
# Resumen final del dataset de Política normalizado
print("=" * 25)
print("   RESUMEN FINAL - DATASET POLÍTICA NORMALIZADO")
print("=" * 25)

# Información básica del dataset
if len(df) > 0:
    fecha_min = df['fecha_posesion'].min()
    fecha_max = df['fecha_posesion'].max()
    periodo_str = f"{fecha_min.date()} a {fecha_max.date()}"
else:
    periodo_str = "Sin datos"

print(f"""
INFORMACIÓN FINAL:
   • Dataset: Política (alcaldes y partidos de Galicia)
   • Registros totales: {len(df):,}
   • Período: {periodo_str}
   • Municipios únicos: {df['municipio'].nunique() if len(df) > 0 else 0}
   • Fechas únicas: {df['fecha_posesion'].nunique() if len(df) > 0 else 0:,}

COLUMNAS FINALES:
""")

for i, col in enumerate(df.columns, 1):
    if len(df) > 0:
        tipo = df[col].dtype
        nulos = df[col].isnull().sum()
        unicos = df[col].nunique()
        print(f"   {i}. {col} ({tipo}) - {unicos:,} únicos, {nulos:,} nulos")
    else:
        print(f"   {i}. {col}")

# Estadísticas específicas de política
if len(df) > 0:
    print(f"\nESTADÍSTICAS POLÍTICAS:")
    print("-" * 40)

    if 'partido' in df.columns:
        partidos_municipales = df['partido'].value_counts()
        print(f"   • Partidos municipales ({len(partidos_municipales)} únicos):")
        for partido, count in partidos_municipales.head(10).items():
            porcentaje = (count / len(df)) * 100
            print(f"     - {partido}: {count:,} registros ({porcentaje:.1f}%)")

    if 'partido_autonomico' in df.columns:
        partidos_autonomicos = df['partido_autonomico'].value_counts()
        print(f"\n   • Partidos autonómicos:")
        for partido, count in partidos_autonomicos.items():
            porcentaje = (count / len(df)) * 100
            print(f"     - {partido}: {count:,} registros ({porcentaje:.1f}%)")

    # Estadísticas de fechas
    print(f"\n   • Análisis temporal:")
    años_unicos = df['fecha_posesion'].dt.year.nunique()
    print(f"     - Años únicos: {años_unicos}")
    
    if 'fecha_baja' in df.columns:
        alcaldes_activos = df['fecha_baja'].isnull().sum()
        print(f"     - Alcaldes aún en cargo: {alcaldes_activos:,}")

# Calidad de datos
if len(df) > 0:
    print(f"""
CALIDAD DE DATOS:
   • Completitud temporal: 100% (período 2000-2022)
   • Cobertura municipal: {df['municipio'].nunique()}/313 municipios de Galicia
   • Datos con partido autonómico: {df['partido_autonomico'].notna().sum():,} registros
   • Densidad promedio: {len(df) / df['fecha_posesion'].nunique():.1f} registros por fecha

ARCHIVO EXPORTADO:
   • Ubicación: {archivo_export}
   • Formato: CSV UTF-8
   • Tamaño: {len(df):,} registros × {len(df.columns)} columnas
   • Columnas: {list(df.columns)}

LISTO PARA:
   • Combinación con dataset de incendios
   • Análisis de correlaciones políticas
   • Modelado predictivo incluyendo variables políticas
   • Análisis de influencia del partido autonómico en políticas municipales
""")

print("=" * 25)
print("   NORMALIZACIÓN COMPLETA FINALIZADA")
print("=" * 25)

   RESUMEN FINAL - DATASET POLÍTICA NORMALIZADO

INFORMACIÓN FINAL:
   • Dataset: Política (alcaldes y partidos de Galicia)
   • Registros totales: 1,784
   • Período: 2000-02-05 a 2022-11-18
   • Municipios únicos: 315
   • Fechas únicas: 222

COLUMNAS FINALES:

   1. municipio (object) - 315 únicos, 0 nulos
   2. partido (object) - 10 únicos, 0 nulos
   3. fecha_posesion (datetime64[ns]) - 222 únicos, 0 nulos
   4. fecha_baja (datetime64[ns]) - 192 únicos, 629 nulos
   5. provincia (object) - 8 únicos, 0 nulos
   6. comunidad_autonoma (object) - 1 únicos, 0 nulos
   7. partido_autonomico (object) - 2 únicos, 0 nulos

ESTADÍSTICAS POLÍTICAS:
----------------------------------------
   • Partidos municipales (10 únicos):
     - PP: 1,020 registros (57.2%)
     - PSOE: 479 registros (26.8%)
     - BNG: 148 registros (8.3%)
     - OTROS: 88 registros (4.9%)
     - IND: 26 registros (1.5%)
     - N.ADS.: 15 registros (0.8%)
     - C. ELECTORAL: 4 registros (0.2%)
     - IU: 2 registros (0

## 3. Expansión temporal completa: Un registro por municipio por día (2000-2022)

In [10]:
# Crear lista completa de municipios gallegos basada en los datos existentes
print("PREPARANDO EXPANSIÓN TEMPORAL COMPLETA")
print("-" * 45)

# Obtener lista única de municipios del dataset político actual
municipios_politica = df['municipio'].unique()
print(f"Municipios en dataset político: {len(municipios_politica)}")

# Cargar también lista de municipios del dataset de incendios para completar
municipios_file = r'C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\01 - municipios\01 - municipios normalizados.csv'

try:
    df_municipios_ref = pd.read_csv(municipios_file)
    if 'municipio' in df_municipios_ref.columns:
        municipios_referencia = df_municipios_ref['municipio'].unique()
        print(f"Municipios en dataset de referencia: {len(municipios_referencia)}")
        
        # Combinar ambas listas para obtener el conjunto completo
        municipios_completos = sorted(set(list(municipios_politica) + list(municipios_referencia)))
        print(f"Total municipios únicos combinados: {len(municipios_completos)}")
    else:
        print("No se encontró columna 'municipio' en el archivo de referencia")
        municipios_completos = sorted(municipios_politica)
        
except FileNotFoundError:
    print("Archivo de municipios de referencia no encontrado, usando solo municipios del dataset político")
    municipios_completos = sorted(municipios_politica)

print(f"\nLista final de municipios: {len(municipios_completos)} municipios")
print(f"Algunos ejemplos: {municipios_completos[:5]}")

# Crear rango completo de fechas (2000-2022)
fecha_inicio = pd.to_datetime('2000-01-01')
fecha_fin = pd.to_datetime('2022-12-31')
fechas_completas = pd.date_range(start=fecha_inicio, end=fecha_fin, freq='D')

print(f"\nRANGO TEMPORAL:")
print(f"   • Fecha inicio: {fecha_inicio.date()}")
print(f"   • Fecha fin: {fecha_fin.date()}")
print(f"   • Total días: {len(fechas_completas):,} días")

# Calcular tamaño del dataset expandido
total_registros = len(municipios_completos) * len(fechas_completas)
print(f"\nTAMAÑO DEL DATASET EXPANDIDO:")
print(f"   • {len(municipios_completos)} municipios × {len(fechas_completas):,} días = {total_registros:,} registros")
print(f"   • Aproximadamente {total_registros/1000000:.1f} millones de registros")

PREPARANDO EXPANSIÓN TEMPORAL COMPLETA
---------------------------------------------
Municipios en dataset político: 315
Municipios en dataset de referencia: 315
Total municipios únicos combinados: 315

Lista final de municipios: 315 municipios
Algunos ejemplos: ['a arnoia', 'a baña', 'a bola', 'a capela', 'a cañiza']

RANGO TEMPORAL:
   • Fecha inicio: 2000-01-01
   • Fecha fin: 2022-12-31
   • Total días: 8,401 días

TAMAÑO DEL DATASET EXPANDIDO:
   • 315 municipios × 8,401 días = 2,646,315 registros
   • Aproximadamente 2.6 millones de registros


In [11]:
# Preparar dataset político optimizado para búsquedas eficientes por municipio y fecha
print("OPTIMIZANDO DATASET POLÍTICO PARA BÚSQUEDAS")
print("-" * 45)

# Crear copia del dataset original con fechas como datetime
df_politica_optimizado = df.copy()

# Asegurar que las fechas estén como datetime
if 'fecha_posesion' in df_politica_optimizado.columns:
    df_politica_optimizado['fecha_posesion'] = pd.to_datetime(df_politica_optimizado['fecha_posesion'])
if 'fecha_baja' in df_politica_optimizado.columns:
    df_politica_optimizado['fecha_baja'] = pd.to_datetime(df_politica_optimizado['fecha_baja'])

# Ordenar por municipio y fecha para búsquedas eficientes
df_politica_optimizado = df_politica_optimizado.sort_values(['municipio', 'fecha_posesion'])

print(f"Dataset político optimizado:")
print(f"   • Registros: {len(df_politica_optimizado):,}")
print(f"   • Ordenado por: municipio, fecha_posesion")

# Función optimizada para encontrar el alcalde vigente en una fecha específica
def encontrar_alcalde_vigente(municipio, fecha_consulta, df_politica=df_politica_optimizado):
    """
    Encuentra el alcalde vigente para un municipio en una fecha específica
    Retorna un diccionario con la información política del alcalde
    
    LÓGICA MEJORADA:
    1. Si la fecha está dentro de un período específico de un alcalde -> ese alcalde
    2. Si la fecha es anterior al primer alcalde registrado -> primer alcalde (asumimos continuidad hacia atrás)
    3. Si la fecha es posterior a un alcalde sin fecha de baja -> ese alcalde (sigue en cargo)
    """
    # Filtrar por municipio y ordenar por fecha de posesión
    registros_municipio = df_politica[df_politica['municipio'] == municipio].sort_values('fecha_posesion')
    
    if len(registros_municipio) == 0:
        return {
            'partido': None,
            'partido_autonomico': None,
            'fecha_posesion': None,
            'fecha_baja': None,
            'encontrado': False
        }
    
    # CASO 1: Buscar alcalde vigente en período específico
    for _, registro in registros_municipio.iterrows():
        fecha_inicio = registro['fecha_posesion']
        fecha_fin = registro.get('fecha_baja', None)
        
        # Verificar si la fecha está en el período del alcalde
        if fecha_inicio <= fecha_consulta:
            # Si no hay fecha de baja o la fecha es anterior/igual a la baja
            if pd.isna(fecha_fin) or fecha_consulta <= fecha_fin:
                return {
                    'partido': registro['partido'],
                    'partido_autonomico': registro['partido_autonomico'],
                    'fecha_posesion': fecha_inicio,
                    'fecha_baja': fecha_fin,
                    'encontrado': True
                }
    
    # CASO 2: Si la fecha es anterior al primer alcalde registrado
    primer_alcalde = registros_municipio.iloc[0]
    if fecha_consulta < primer_alcalde['fecha_posesion']:
        # Asumir que el primer alcalde registrado estaba en cargo desde antes
        return {
            'partido': primer_alcalde['partido'],
            'partido_autonomico': primer_alcalde['partido_autonomico'],
            'fecha_posesion': primer_alcalde['fecha_posesion'],
            'fecha_baja': primer_alcalde.get('fecha_baja', None),
            'encontrado': True
        }
    
    # CASO 3: Si la fecha es posterior, buscar el último alcalde sin fecha de baja
    for _, registro in registros_municipio.iterrows():
        if pd.isna(registro.get('fecha_baja', None)):
            # Este alcalde sigue en cargo
            return {
                'partido': registro['partido'],
                'partido_autonomico': registro['partido_autonomico'],
                'fecha_posesion': registro['fecha_posesion'],
                'fecha_baja': registro.get('fecha_baja', None),
                'encontrado': True
            }
    
    # Si no se encuentra alcalde vigente (caso muy raro)
    return {
        'partido': None,
        'partido_autonomico': None,
        'fecha_posesion': None,
        'fecha_baja': None,
        'encontrado': False
    }

# Probar la función con algunos ejemplos
print(f"\nPRUEBAS DE LA FUNCIÓN:")
print("-" * 30)

municipios_prueba = municipios_completos[:3]  # Primeros 3 municipios
fechas_prueba = [pd.to_datetime('2000-06-15'), pd.to_datetime('2010-03-10'), pd.to_datetime('2020-09-05')]

for municipio in municipios_prueba:
    for fecha in fechas_prueba:
        resultado = encontrar_alcalde_vigente(municipio, fecha)
        estado = "ok" if resultado['encontrado'] else "error"
        partido = resultado['partido'] if resultado['encontrado'] else "No encontrado"
        print(f"   {estado} {municipio} en {fecha.date()}: {partido}")

print(f"\nFunción de búsqueda optimizada lista")

OPTIMIZANDO DATASET POLÍTICO PARA BÚSQUEDAS
---------------------------------------------
Dataset político optimizado:
   • Registros: 1,784
   • Ordenado por: municipio, fecha_posesion

PRUEBAS DE LA FUNCIÓN:
------------------------------
   ok a arnoia en 2000-06-15: PP
   ok a arnoia en 2010-03-10: PP
   ok a arnoia en 2020-09-05: PP
   ok a baña en 2000-06-15: PP
   ok a baña en 2010-03-10: PP
   ok a baña en 2020-09-05: PP
   ok a bola en 2000-06-15: PP
   ok a bola en 2010-03-10: IND
   ok a bola en 2020-09-05: IND

Función de búsqueda optimizada lista


In [12]:
# Diagnóstico detallado de los municipios que anteriormente daban problemas
print("DIAGNÓSTICO DETALLADO DE MUNICIPIOS PROBLEMÁTICOS")
print("-" * 55)

# Municipios que dieron problemas en las pruebas anteriores
municipios_diagnostico = ['a arnoia', 'a baña', 'a bola']

for municipio in municipios_diagnostico:
    print(f"\nMUNICIPIO: {municipio}")
    print("-" * 30)
    
    # Buscar todos los registros de este municipio
    registros = df_politica_optimizado[df_politica_optimizado['municipio'] == municipio]
    
    if len(registros) == 0:
        print("   No se encontraron registros para este municipio")
        continue
    
    print(f"   Total registros: {len(registros)}")
    
    # Mostrar todos los períodos
    for i, (_, registro) in enumerate(registros.iterrows(), 1):
        fecha_inicio = registro['fecha_posesion']
        fecha_fin = registro.get('fecha_baja', None)
        partido = registro['partido']
        
        if pd.isna(fecha_fin):
            print(f"   {i}. {partido}: {fecha_inicio.date()} → [SIN FECHA BAJA]")
        else:
            print(f"   {i}. {partido}: {fecha_inicio.date()} → {fecha_fin.date()}")
    
    # Probar la función mejorada con las fechas problemáticas
    fechas_test = [pd.to_datetime('2000-06-15'), pd.to_datetime('2010-03-10'), pd.to_datetime('2020-09-05')]
    
    print(f"\n   PRUEBAS CON FUNCIÓN MEJORADA:")
    for fecha in fechas_test:
        resultado = encontrar_alcalde_vigente(municipio, fecha)
        estado = "ok" if resultado['encontrado'] else "error"
        partido = resultado['partido'] if resultado['encontrado'] else "No encontrado"
        
        # Información adicional sobre la lógica aplicada
        if resultado['encontrado']:
            posesion = resultado['fecha_posesion'].date() if resultado['fecha_posesion'] else "N/A"
            print(f"      {estado} {fecha.date()}: {partido} (posesión: {posesion})")
        else:
            print(f"      {estado} {fecha.date()}: {partido}")

print(f"\nDiagnóstico completado - Función mejorada debería resolver los problemas anteriores")

DIAGNÓSTICO DETALLADO DE MUNICIPIOS PROBLEMÁTICOS
-------------------------------------------------------

MUNICIPIO: a arnoia
------------------------------
   Total registros: 6
   1. PP: 2003-06-14 → [SIN FECHA BAJA]
   2. PP: 2007-11-17 → 2009-06-02
   3. PP: 2009-06-02 → 2011-06-11
   4. PP: 2011-06-11 → 2015-06-13
   5. PP: 2015-06-13 → 2019-06-15
   6. PP: 2019-06-15 → [SIN FECHA BAJA]

   PRUEBAS CON FUNCIÓN MEJORADA:
      ok 2000-06-15: PP (posesión: 2003-06-14)
      ok 2010-03-10: PP (posesión: 2003-06-14)
      ok 2020-09-05: PP (posesión: 2003-06-14)

MUNICIPIO: a baña
------------------------------
   Total registros: 6
   1. PP: 2003-06-14 → [SIN FECHA BAJA]
   2. PP: 2007-06-16 → 2011-06-11
   3. PP: 2011-06-11 → 2015-06-13
   4. PP: 2015-06-13 → 2019-06-15
   5. PSOE: 2019-06-15 → 2021-12-12
   6. PP: 2021-12-14 → [SIN FECHA BAJA]

   PRUEBAS CON FUNCIÓN MEJORADA:
      ok 2000-06-15: PP (posesión: 2003-06-14)
      ok 2010-03-10: PP (posesión: 2003-06-14)
      ok 20

In [13]:
# Crear dataset expandido procesando por lotes para optimizar memoria
print("CREANDO DATASET EXPANDIDO (PROCESAMIENTO POR LOTES)")
print("-" * 55)

import gc  # Para gestión de memoria

# Configuración de procesamiento por lotes
BATCH_SIZE = 50  # Procesar 50 municipios a la vez
municipios_totales = len(municipios_completos)
fechas_totales = len(fechas_completas)

print(f"CONFIGURACIÓN:")
print(f"   • Municipios totales: {municipios_totales}")
print(f"   • Fechas totales: {fechas_totales:,}")
print(f"   • Tamaño de lote: {BATCH_SIZE} municipios")
print(f"   • Número de lotes: {(municipios_totales + BATCH_SIZE - 1) // BATCH_SIZE}")

# Lista para almacenar los DataFrames de cada lote
lotes_resultados = []

# Procesar municipios en lotes
for i in range(0, municipios_totales, BATCH_SIZE):
    lote_numero = (i // BATCH_SIZE) + 1
    municipios_lote = municipios_completos[i:i + BATCH_SIZE]
    
    print(f"\nProcesando lote {lote_numero}/{(municipios_totales + BATCH_SIZE - 1) // BATCH_SIZE}")
    print(f"   Municipios {i+1}-{min(i+BATCH_SIZE, municipios_totales)}: {len(municipios_lote)} municipios")
    
    # Crear combinación cartesiana: municipios × fechas para este lote
    lote_data = []
    
    for municipio in municipios_lote:
        for fecha in fechas_completas:
            # Encontrar información política para esta fecha
            info_politica = encontrar_alcalde_vigente(municipio, fecha)
            
            # Crear registro
            registro = {
                'municipio': municipio,
                'fecha': fecha,
                'partido': info_politica['partido'],
                'partido_autonomico': info_politica['partido_autonomico'],
                'fecha_posesion': info_politica['fecha_posesion'],
                'fecha_baja': info_politica['fecha_baja']
            }
            lote_data.append(registro)
    
    # Convertir a DataFrame
    df_lote = pd.DataFrame(lote_data)
    lotes_resultados.append(df_lote)
    
    # Estadísticas del lote
    registros_lote = len(df_lote)
    registros_con_partido = df_lote['partido'].notna().sum()
    cobertura_lote = (registros_con_partido / registros_lote) * 100
    
    print(f"   Registros creados: {registros_lote:,}")
    print(f"   Con información política: {registros_con_partido:,} ({cobertura_lote:.1f}%)")
    
    # Limpiar memoria cada ciertos lotes
    if lote_numero % 5 == 0:
        gc.collect()

print(f"\nCOMBINANDO TODOS LOS LOTES...")

# Combinar todos los lotes en un solo DataFrame
df_expandido = pd.concat(lotes_resultados, ignore_index=True)

# Limpiar memoria
del lotes_resultados
gc.collect()

print(f"Dataset expandido creado exitosamente:")
print(f"   Total registros: {len(df_expandido):,}")
print(f"   Período: {df_expandido['fecha'].min().date()} a {df_expandido['fecha'].max().date()}")
print(f"   Municipios únicos: {df_expandido['municipio'].nunique()}")
print(f"   Fechas únicas: {df_expandido['fecha'].nunique():,}")

CREANDO DATASET EXPANDIDO (PROCESAMIENTO POR LOTES)
-------------------------------------------------------
CONFIGURACIÓN:
   • Municipios totales: 315
   • Fechas totales: 8,401
   • Tamaño de lote: 50 municipios
   • Número de lotes: 7

Procesando lote 1/7
   Municipios 1-50: 50 municipios
   Registros creados: 420,050
   Con información política: 420,050 (100.0%)

Procesando lote 2/7
   Municipios 51-100: 50 municipios
   Registros creados: 420,050
   Con información política: 420,050 (100.0%)

Procesando lote 3/7
   Municipios 101-150: 50 municipios
   Registros creados: 420,050
   Con información política: 420,050 (100.0%)

Procesando lote 4/7
   Municipios 151-200: 50 municipios
   Registros creados: 420,050
   Con información política: 420,050 (100.0%)

Procesando lote 5/7
   Municipios 201-250: 50 municipios
   Registros creados: 420,050
   Con información política: 420,050 (100.0%)

Procesando lote 6/7
   Municipios 251-300: 50 municipios
   Registros creados: 420,050
   Con i

In [14]:
# Análisis de calidad y estadísticas del dataset expandido
print("ANÁLISIS DE CALIDAD DEL DATASET EXPANDIDO")
print("-" * 50)

# Estadísticas básicas
total_registros = len(df_expandido)
registros_con_partido = df_expandido['partido'].notna().sum()
registros_sin_partido = df_expandido['partido'].isnull().sum()
cobertura_total = (registros_con_partido / total_registros) * 100

print(f"ESTADÍSTICAS GENERALES:")
print(f"   • Total registros: {total_registros:,}")
print(f"   • Con información política: {registros_con_partido:,} ({cobertura_total:.1f}%)")
print(f"   • Sin información política: {registros_sin_partido:,} ({100-cobertura_total:.1f}%)")

# Verificar completitud temporal
print(f"\nCOMPLETITUD TEMPORAL:")
fechas_esperadas = len(fechas_completas)
fechas_reales = df_expandido['fecha'].nunique()
municipios_esperados = len(municipios_completos)
municipios_reales = df_expandido['municipio'].nunique()

print(f"   • Fechas esperadas: {fechas_esperadas:,}")
print(f"   • Fechas reales: {fechas_reales:,}")
print(f"   • Municipios esperados: {municipios_esperados}")
print(f"   • Municipios reales: {municipios_reales}")

completitud_fechas = (fechas_reales / fechas_esperadas) * 100
completitud_municipios = (municipios_reales / municipios_esperados) * 100

print(f"   • Completitud fechas: {completitud_fechas:.1f}%")
print(f"   • Completitud municipios: {completitud_municipios:.1f}%")

# Análisis por año
print(f"\nCOBERTURA POR AÑO:")
print("-" * 25)
df_expandido['año'] = df_expandido['fecha'].dt.year
cobertura_anual = df_expandido.groupby('año').agg({
    'partido': lambda x: x.notna().sum(),
    'municipio': 'count'
}).reset_index()
cobertura_anual['porcentaje'] = (cobertura_anual['partido'] / cobertura_anual['municipio']) * 100

for _, row in cobertura_anual.iterrows():
    año = int(row['año'])
    pct = row['porcentaje']
    registros_año = row['municipio']
    print(f"   {año}: {pct:5.1f}% ({row['partido']:,}/{registros_año:,} registros)")

# Top municipios con mejor cobertura
print(f"\nTOP 10 MUNICIPIOS CON MEJOR COBERTURA:")
print("-" * 45)
cobertura_municipal = df_expandido.groupby('municipio').agg({
    'partido': lambda x: x.notna().sum(),
    'fecha': 'count'
}).reset_index()
cobertura_municipal['porcentaje'] = (cobertura_municipal['partido'] / cobertura_municipal['fecha']) * 100
cobertura_municipal = cobertura_municipal.sort_values('porcentaje', ascending=False)

for i, (_, row) in enumerate(cobertura_municipal.head(10).iterrows(), 1):
    municipio = row['municipio']
    pct = row['porcentaje']
    print(f"   {i:2d}. {municipio}: {pct:.1f}%")

# Municipios sin información política
municipios_sin_info = cobertura_municipal[cobertura_municipal['porcentaje'] == 0]['municipio'].tolist()
if municipios_sin_info:
    print(f"\nMUNICIPIOS SIN INFORMACIÓN POLÍTICA ({len(municipios_sin_info)}):")
    print(f"   {municipios_sin_info}")

# Distribución de partidos en el dataset expandido
print(f"\nDISTRIBUCIÓN DE PARTIDOS (REGISTROS CON DATOS):")
print("-" * 50)
if registros_con_partido > 0:
    dist_partidos = df_expandido['partido'].value_counts()
    for partido, count in dist_partidos.head(10).items():
        porcentaje = (count / registros_con_partido) * 100
        print(f"   • {partido}: {count:,} registros ({porcentaje:.1f}%)")
    
    if len(dist_partidos) > 10:
        otros = dist_partidos.iloc[10:].sum()
        pct_otros = (otros / registros_con_partido) * 100
        print(f"   • Otros ({len(dist_partidos)-10} partidos): {otros:,} registros ({pct_otros:.1f}%)")

# Distribución de partidos autonómicos
print(f"\nDISTRIBUCIÓN DE PARTIDOS AUTONÓMICOS:")
print("-" * 40)
if 'partido_autonomico' in df_expandido.columns:
    dist_aut = df_expandido['partido_autonomico'].value_counts()
    for partido, count in dist_aut.items():
        porcentaje = (count / total_registros) * 100
        print(f"   • {partido}: {count:,} registros ({porcentaje:.1f}%)")

# Eliminar columna temporal
df_expandido.drop('año', axis=1, inplace=True)

print(f"\nAnálisis de calidad completado")

ANÁLISIS DE CALIDAD DEL DATASET EXPANDIDO
--------------------------------------------------
ESTADÍSTICAS GENERALES:
   • Total registros: 2,646,315
   • Con información política: 2,646,315 (100.0%)
   • Sin información política: 0 (0.0%)

COMPLETITUD TEMPORAL:
   • Fechas esperadas: 8,401
   • Fechas reales: 8,401
   • Municipios esperados: 315
   • Municipios reales: 315
   • Completitud fechas: 100.0%
   • Completitud municipios: 100.0%

COBERTURA POR AÑO:
-------------------------
   2000: 100.0% (115,290.0/115,290.0 registros)
   2001: 100.0% (114,975.0/114,975.0 registros)
   2002: 100.0% (114,975.0/114,975.0 registros)
   2003: 100.0% (114,975.0/114,975.0 registros)
   2004: 100.0% (115,290.0/115,290.0 registros)
   2005: 100.0% (114,975.0/114,975.0 registros)
   2006: 100.0% (114,975.0/114,975.0 registros)
   2007: 100.0% (114,975.0/114,975.0 registros)
   2008: 100.0% (115,290.0/115,290.0 registros)
   2009: 100.0% (114,975.0/114,975.0 registros)
   2010: 100.0% (114,975.0/114

In [15]:
# Preparar y exportar el dataset expandido final
print("PREPARANDO EXPORTACIÓN DEL DATASET EXPANDIDO")
print("-" * 50)

# Crear copia para exportación
df_export_expandido = df_expandido.copy()

# Ordenar por municipio y fecha para mejor organización
df_export_expandido = df_export_expandido.sort_values(['municipio', 'fecha'])

# Agregar columnas útiles para análisis
df_export_expandido['año'] = df_export_expandido['fecha'].dt.year
df_export_expandido['mes'] = df_export_expandido['fecha'].dt.month
df_export_expandido['dia'] = df_export_expandido['fecha'].dt.day
df_export_expandido['dia_año'] = df_export_expandido['fecha'].dt.dayofyear

# Convertir fechas a formato string para compatibilidad
df_export_expandido['fecha'] = df_export_expandido['fecha'].dt.strftime('%Y-%m-%d')

# Manejar fechas de posesión y baja
if 'fecha_posesion' in df_export_expandido.columns:
    df_export_expandido['fecha_posesion'] = df_export_expandido['fecha_posesion'].dt.strftime('%Y-%m-%d').fillna('')

if 'fecha_baja' in df_export_expandido.columns:
    df_export_expandido['fecha_baja'] = df_export_expandido['fecha_baja'].dt.strftime('%Y-%m-%d').fillna('')

# Reordenar columnas para mejor legibilidad
columnas_ordenadas = [
    'municipio', 'fecha', 'año', 'mes', 'dia', 'dia_año',
    'partido', 'partido_autonomico', 'fecha_posesion', 'fecha_baja'
]

# Filtrar solo las columnas que existen
columnas_finales = [col for col in columnas_ordenadas if col in df_export_expandido.columns]
df_export_expandido = df_export_expandido[columnas_finales]

print(f"ESTRUCTURA FINAL DEL DATASET:")
print(f"   • Registros: {len(df_export_expandido):,}")
print(f"   • Columnas: {len(df_export_expandido.columns)}")
print(f"   • Ordenado por: municipio, fecha")

for i, col in enumerate(df_export_expandido.columns, 1):
    print(f"   {i:2d}. {col}")

# Verificar tamaño del archivo antes de guardar
tamaño_estimado_mb = (len(df_export_expandido) * len(df_export_expandido.columns) * 10) / (1024 * 1024)  # Estimación
print(f"\nESTIMACIÓN DE TAMAÑO:")
print(f"   • Tamaño estimado: ~{tamaño_estimado_mb:.1f} MB")

if tamaño_estimado_mb > 500:
    print(f"   ADVERTENCIA: Archivo grande (>{tamaño_estimado_mb:.0f} MB)")
    print(f"   Considera comprimir o dividir el archivo si es necesario")

# Crear directorio de destino
import os
ruta_export_expandido = r'C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\08 - Politica'
os.makedirs(ruta_export_expandido, exist_ok=True)

# Definir archivo de salida
archivo_expandido = os.path.join(ruta_export_expandido, '02 - politica normalizado completo expandido.csv')

print(f"\nGUARDANDO ARCHIVO...")
print(f"   Ubicación: {archivo_expandido}")

# Guardar el dataset expandido
df_export_expandido.to_csv(archivo_expandido, index=False, encoding='utf-8')

# Verificar que se guardó correctamente
if os.path.exists(archivo_expandido):
    tamaño_real = os.path.getsize(archivo_expandido)
    tamaño_real_mb = tamaño_real / (1024 * 1024)
    
    print(f"ARCHIVO GUARDADO EXITOSAMENTE:")
    print(f"   • Tamaño real: {tamaño_real_mb:.1f} MB ({tamaño_real:,} bytes)")
    print(f"   • Formato: CSV UTF-8")
    print(f"   • Registros: {len(df_export_expandido):,}")
    print(f"   • Período: {df_export_expandido['fecha'].min()} a {df_export_expandido['fecha'].max()}")
else:
    print("ERROR: No se pudo guardar el archivo")

print(f"\nARCHIVO LISTO PARA:")
print(f"   • Combinación directa con otros datasets temporales")
print(f"   • Análisis de series temporales políticas")
print(f"   • Modelado predictivo con variables políticas completas")
print(f"   • Estudios de correlación temporal entre política e incendios")

PREPARANDO EXPORTACIÓN DEL DATASET EXPANDIDO
--------------------------------------------------
ESTRUCTURA FINAL DEL DATASET:
   • Registros: 2,646,315
   • Columnas: 10
   • Ordenado por: municipio, fecha
    1. municipio
    2. fecha
    3. año
    4. mes
    5. dia
    6. dia_año
    7. partido
    8. partido_autonomico
    9. fecha_posesion
   10. fecha_baja

ESTIMACIÓN DE TAMAÑO:
   • Tamaño estimado: ~252.4 MB

GUARDANDO ARCHIVO...
   Ubicación: C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\08 - Politica\02 - politica normalizado completo expandido.csv
ARCHIVO GUARDADO EXITOSAMENTE:
   • Tamaño real: 137.9 MB (144,580,367 bytes)
   • Formato: CSV UTF-8
   • Registros: 2,646,315
   • Período: 2000-01-01 a 2022-12-31

ARCHIVO LISTO PARA:
   • Combinación directa con otros datasets temporales
   • Análisis de series temporales políticas
   • Modelado predictivo con variables políticas completas
   • Estudios de correlación temporal entre política e incendio

In [16]:
# Resumen final con comparación de ambos datasets creados
print("=" * 30)
print("   RESUMEN FINAL - DATASETS POLÍTICA NORMALIZADOS")
print("=" * 30)

print(f"""
COMPARACIÓN DE DATASETS CREADOS:

1️DATASET ORIGINAL NORMALIZADO:
   • Archivo: normalizado completo politica.csv
   • Registros: {len(df):,}
   • Estructura: Un registro por alcalde por período
   • Período: 2000-2022 (solo fechas de posesión)
   • Uso: Análisis de cambios políticos y mandatos

2️DATASET EXPANDIDO TEMPORAL:
   • Archivo: normalizado_completo_politica_expandido.csv
   • Registros: {len(df_export_expandido):,}
   • Estructura: Un registro por municipio por día
   • Período: 2000-2022 (cobertura completa diaria)
   • Uso: Series temporales y combinación con otros datasets

VENTAJAS DEL DATASET EXPANDIDO:
   Compatibilidad perfecta con datasets de incendios y clima
   Facilita análisis de correlaciones temporales
   Permite agregaciones por cualquier período (mensual, anual, etc.)
   Simplifica joins por municipio+fecha
   Mantiene información política estable para cada día

COLUMNAS DISPONIBLES EN DATASET EXPANDIDO:
""")

for i, col in enumerate(df_export_expandido.columns, 1):
    print(f"   {i:2d}. {col}")

# Calcular algunas estadísticas finales útiles
municipios_con_info = df_export_expandido[df_export_expandido['partido'].notna()]['municipio'].nunique()
total_municipios = df_export_expandido['municipio'].nunique()
cobertura_municipal = (municipios_con_info / total_municipios) * 100

años_unicos = df_export_expandido['año'].nunique()
fechas_unicas = df_export_expandido['fecha'].nunique()

print(f"""
ESTADÍSTICAS CLAVE DEL DATASET EXPANDIDO:
   • Municipios con información política: {municipios_con_info}/{total_municipios} ({cobertura_municipal:.1f}%)
   • Años cubiertos: {años_unicos} años (2000-2022)
   • Fechas únicas: {fechas_unicas:,} días
   • Densidad promedia: {len(df_export_expandido)/fechas_unicas:.1f} municipios por día

PRÓXIMOS PASOS RECOMENDADOS:
   1. Combinar con dataset de incendios normalizado
   2. Agregar variables meteorológicas
   3. Crear dataset maestro para modelado
   4. Análisis de correlaciones política-incendios

UBICACIONES DE ARCHIVOS:
   • Dataset original: C:\\00 - Proyecto Incendios Galicia 0.0\\data\\03 - Normalizado completo\\08 - Politica\\normalizado completo politica.csv
   • Dataset expandido: C:\\00 - Proyecto Incendios Galicia 0.0\\data\\03 - Normalizado completo\\08 - Politica\\normalizado_completo_politica_expandido.csv

NORMALIZACIÓN POLÍTICA COMPLETADA CON ÉXITO
   Ambos datasets están listos para la siguiente fase del proyecto
""")

print("=" * 30)
print("   PROCESO FINALIZADO - DATASETS POLÍTICA LISTOS")
print("=" * 30)

   RESUMEN FINAL - DATASETS POLÍTICA NORMALIZADOS

COMPARACIÓN DE DATASETS CREADOS:

1️DATASET ORIGINAL NORMALIZADO:
   • Archivo: normalizado completo politica.csv
   • Registros: 1,784
   • Estructura: Un registro por alcalde por período
   • Período: 2000-2022 (solo fechas de posesión)
   • Uso: Análisis de cambios políticos y mandatos

2️DATASET EXPANDIDO TEMPORAL:
   • Archivo: normalizado_completo_politica_expandido.csv
   • Registros: 2,646,315
   • Estructura: Un registro por municipio por día
   • Período: 2000-2022 (cobertura completa diaria)
   • Uso: Series temporales y combinación con otros datasets

VENTAJAS DEL DATASET EXPANDIDO:
   Compatibilidad perfecta con datasets de incendios y clima
   Facilita análisis de correlaciones temporales
   Permite agregaciones por cualquier período (mensual, anual, etc.)
   Simplifica joins por municipio+fecha
   Mantiene información política estable para cada día

COLUMNAS DISPONIBLES EN DATASET EXPANDIDO:

    1. municipio
    2. fech

In [17]:
# Optimización final: Limpiar columnas y renombrar para versión definitiva
print("OPTIMIZACIÓN FINAL DEL DATASET EXPANDIDO")
print("-" * 50)

# Crear copia del dataset expandido para optimización final
df_final_optimizado = df_expandido.copy()

print(f"DATASET ANTES DE OPTIMIZACIÓN:")
print(f"   • Registros: {len(df_final_optimizado):,}")
print(f"   • Columnas: {len(df_final_optimizado.columns)}")
print(f"   • Columnas actuales: {list(df_final_optimizado.columns)}")

# 1. RENOMBRAR COLUMNAS PARA MAYOR CLARIDAD
print(f"\nRENOMBRANDO COLUMNAS:")
print("-" * 25)

# Mapeo de renombrado
renombrado_columnas = {
    'partido': 'partido_municipal',
    'partido_autonomico': 'partido_autonomico'  # Se mantiene igual
}

# Aplicar renombrado solo a las columnas que existen
for original, nuevo in renombrado_columnas.items():
    if original in df_final_optimizado.columns:
        df_final_optimizado.rename(columns={original: nuevo}, inplace=True)
        print(f"   '{original}' → '{nuevo}'")
    else:
        print(f"   Columna '{original}' no encontrada")

# 2. PREPARAR COLUMNAS TEMPORALES OPTIMIZADAS
print(f"\nOPTIMIZANDO COLUMNAS TEMPORALES:")
print("-" * 35)

# Convertir fecha a string si no lo está ya
if df_final_optimizado['fecha'].dtype != 'object':
    df_final_optimizado['fecha'] = df_final_optimizado['fecha'].dt.strftime('%Y-%m-%d')
    print("   Fecha convertida a string formato YYYY-MM-DD")

# Convertir fechas de posesión y baja si existen
if 'fecha_posesion' in df_final_optimizado.columns:
    if df_final_optimizado['fecha_posesion'].dtype != 'object':
        df_final_optimizado['fecha_posesion'] = df_final_optimizado['fecha_posesion'].dt.strftime('%Y-%m-%d').fillna('')
    print("   fecha_posesion optimizada")

if 'fecha_baja' in df_final_optimizado.columns:
    if df_final_optimizado['fecha_baja'].dtype != 'object':
        df_final_optimizado['fecha_baja'] = df_final_optimizado['fecha_baja'].dt.strftime('%Y-%m-%d').fillna('')
    print("   fecha_baja optimizada")

# 3. SELECCIONAR SOLO LAS COLUMNAS NECESARIAS (SIN AÑO, MES, DIA INDIVIDUALES)
print(f"\nSELECCIONANDO COLUMNAS FINALES:")
print("-" * 35)

# Columnas que queremos mantener (sin columnas temporales individuales)
columnas_finales_optimizadas = [
    'municipio',
    'fecha',  # Solo esta columna temporal
    'partido_municipal',
    'partido_autonomico',
    'fecha_posesion',
    'fecha_baja'
]

# Filtrar solo las columnas que realmente existen
columnas_disponibles_final = [col for col in columnas_finales_optimizadas if col in df_final_optimizado.columns]
columnas_eliminadas = [col for col in df_final_optimizado.columns if col not in columnas_disponibles_final]

# Aplicar selección de columnas
df_final_optimizado = df_final_optimizado[columnas_disponibles_final]

print(f"   Columnas mantenidas ({len(columnas_disponibles_final)}):")
for i, col in enumerate(columnas_disponibles_final, 1):
    print(f"      {i}. {col}")

if columnas_eliminadas:
    print(f"   Columnas eliminadas: {columnas_eliminadas}")

# 4. ORDENAR Y VERIFICAR ESTRUCTURA FINAL
print(f"\nESTRUCTURA FINAL OPTIMIZADA:")
print("-" * 35)

# Ordenar por municipio y fecha
df_final_optimizado = df_final_optimizado.sort_values(['municipio', 'fecha'])

print(f"   • Registros: {len(df_final_optimizado):,}")
print(f"   • Columnas: {len(df_final_optimizado.columns)}")
print(f"   • Período: {df_final_optimizado['fecha'].min()} a {df_final_optimizado['fecha'].max()}")
print(f"   • Municipios únicos: {df_final_optimizado['municipio'].nunique()}")

# Verificar calidad de datos
registros_con_partido = df_final_optimizado['partido_municipal'].notna().sum()
cobertura_politica = (registros_con_partido / len(df_final_optimizado)) * 100

print(f"   • Cobertura política: {cobertura_politica:.1f}% ({registros_con_partido:,} registros)")

print(f"\nOptimización completada - Dataset listo para exportación final")

OPTIMIZACIÓN FINAL DEL DATASET EXPANDIDO
--------------------------------------------------
DATASET ANTES DE OPTIMIZACIÓN:
   • Registros: 2,646,315
   • Columnas: 6
   • Columnas actuales: ['municipio', 'fecha', 'partido', 'partido_autonomico', 'fecha_posesion', 'fecha_baja']

RENOMBRANDO COLUMNAS:
-------------------------
   'partido' → 'partido_municipal'
   'partido_autonomico' → 'partido_autonomico'

OPTIMIZANDO COLUMNAS TEMPORALES:
-----------------------------------
   Fecha convertida a string formato YYYY-MM-DD
   fecha_posesion optimizada
   fecha_baja optimizada

SELECCIONANDO COLUMNAS FINALES:
-----------------------------------
   Columnas mantenidas (6):
      1. municipio
      2. fecha
      3. partido_municipal
      4. partido_autonomico
      5. fecha_posesion
      6. fecha_baja

ESTRUCTURA FINAL OPTIMIZADA:
-----------------------------------
   • Registros: 2,646,315
   • Columnas: 6
   • Período: 2000-01-01 a 2022-12-31
   • Municipios únicos: 315
   • Cobertura

In [18]:
# Guardar la versión final optimizada del dataset expandido
print("GUARDANDO VERSIÓN FINAL OPTIMIZADA")
print("-" * 40)

import os

# Crear directorio de destino si no existe
ruta_final = r'C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\08 - Politica'
os.makedirs(ruta_final, exist_ok=True)

# Definir nombre del archivo final optimizado
archivo_final = os.path.join(ruta_final, '03 - politica expandido final.csv')

# Guardar el dataset optimizado
df_final_optimizado.to_csv(archivo_final, index=False, encoding='utf-8')

# Verificar que se guardó correctamente
if os.path.exists(archivo_final):
    tamaño_archivo = os.path.getsize(archivo_final)
    tamaño_mb = tamaño_archivo / (1024 * 1024)
    
    print(f"ARCHIVO FINAL GUARDADO EXITOSAMENTE:")
    print(f"   Ubicación: {archivo_final}")
    print(f"   Registros: {len(df_final_optimizado):,}")
    print(f"   Columnas: {len(df_final_optimizado.columns)}")
    print(f"   Tamaño: {tamaño_mb:.1f} MB ({tamaño_archivo:,} bytes)")
    print(f"   Período: {df_final_optimizado['fecha'].min()} a {df_final_optimizado['fecha'].max()}")
    
    print(f"\nESTRUCTURA FINAL:")
    for i, col in enumerate(df_final_optimizado.columns, 1):
        valores_unicos = df_final_optimizado[col].nunique()
        nulos = df_final_optimizado[col].isnull().sum()
        print(f"   {i}. {col}: {valores_unicos:,} únicos, {nulos:,} nulos")
    
    print(f"\nCARACTERÍSTICAS DEL DATASET FINAL:")
    print(f"   Nomenclatura clara: 'partido_municipal' vs 'partido_autonomico'")
    print(f"   Solo columna 'fecha' (sin año/mes/día separados)")
    print(f"   Cobertura temporal completa: 8,401 días × municipios")
    print(f"   Optimizado para joins con otros datasets")
    print(f"   Listo para análisis de series temporales")
    
    # Mostrar muestra de datos
    print(f"\nMUESTRA DE DATOS (PRIMERAS 3 FILAS):")
    print(df_final_optimizado.head(3).to_string(index=False))
    
else:
    print("ERROR: No se pudo guardar el archivo final")

print(f"\nPROCESO COMPLETADO - DATASET POLÍTICO EXPANDIDO FINALIZADO")
print(f"Archivo listo en: {archivo_final}")

GUARDANDO VERSIÓN FINAL OPTIMIZADA
----------------------------------------
ARCHIVO FINAL GUARDADO EXITOSAMENTE:
   Ubicación: C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\08 - Politica\03 - politica expandido final.csv
   Registros: 2,646,315
   Columnas: 6
   Tamaño: 103.4 MB (108,430,325 bytes)
   Período: 2000-01-01 a 2022-12-31

ESTRUCTURA FINAL:
   1. municipio: 315 únicos, 0 nulos
   2. fecha: 8,401 únicos, 0 nulos
   3. partido_municipal: 5 únicos, 0 nulos
   4. partido_autonomico: 2 únicos, 0 nulos
   5. fecha_posesion: 56 únicos, 0 nulos
   6. fecha_baja: 34 únicos, 0 nulos

CARACTERÍSTICAS DEL DATASET FINAL:
   Nomenclatura clara: 'partido_municipal' vs 'partido_autonomico'
   Solo columna 'fecha' (sin año/mes/día separados)
   Cobertura temporal completa: 8,401 días × municipios
   Optimizado para joins con otros datasets
   Listo para análisis de series temporales

MUESTRA DE DATOS (PRIMERAS 3 FILAS):
municipio      fecha partido_municipal partid